# Python Environment Setup

## Overview

This course supports two ways to run labs. Choose the one that fits your setup.

**Option A (Local): pixi + VSCode or JupyterLab** — Recommended for consistent, reproducible environments. All dependencies are pinned via a lock file, so your environment matches every other student's exactly. Requires a one-time setup on your machine.

**Option B (Cloud): Google Colab** — No installation needed. Open any lab notebook directly in your browser using the Colab badge at the top of each notebook. Best if you do not want to manage a local environment.

> #### 💡 Recommendation
>
> We recommend **Option A** for all students. The pixi environment guarantees that your code runs identically to the instructor's setup, which makes debugging and grading straightforward. If your machine has an NVIDIA GPU, the local environment is configured to use it automatically via CUDA.
>
> Option B is a convenient fallback if you do not want to manage a local environment, or if you need access to a GPU but do not have one on your machine. Note that Colab sessions are ephemeral and require re-running setup cells every time you reconnect.

Both options provide access to all required packages. The sections below walk through each option in detail.

## Option A (Local): pixi + VSCode or JupyterLab

### Why pixi?

The PyTorch team discontinued distribution via the Anaconda `pytorch` channel in November 2024. The recommended approach is now conda-forge or PyPI, both of which pixi handles natively.

Compared to conda or mamba, pixi offers three main advantages for this course. It is significantly faster at solving and installing environments. It uses a lock file (`pixi.lock`) that pins every dependency to an exact version, so every student gets an identical environment. It also handles the common conflict between conda-managed and pip-managed packages in a single unified workflow.

### Prerequisites

Before starting, install the following tools on your machine.

> #### ❗ Windows Users
>
> This course environment supports **Linux** and **macOS** only. Windows users must use **WSL2 (Windows Subsystem for Linux)** to run in a Linux environment. Install WSL2 first, then follow the Linux instructions below inside your WSL2 terminal.
>
> 1. Open PowerShell as Administrator and run:
> ```powershell
> wsl --install
> ```
> 2. Restart your computer when prompted.
> 3. Open the **Ubuntu** app from the Start menu to finish setup (username and password).
> 4. All remaining steps in this guide should be run inside this Ubuntu terminal.
>
> For the full WSL2 installation guide, see <https://learn.microsoft.com/en-us/windows/wsl/install>.

**git** — Required for cloning the course repository.

- **Linux (WSL2 included)**: `sudo apt update && sudo apt install git`
- **macOS**: Verify with `git --version`. If not installed, macOS will prompt you to install Xcode Command Line Tools automatically, or run `xcode-select --install`.

**pixi** — A fast, cross-platform package manager. `sudo` is not required — pixi installs to `~/.pixi/` under your own user account, and each project environment lives inside the project folder under `.pixi/envs/`. For the full installation guide, see <https://pixi.prefix.dev/latest/installation>.

```bash
# Choose one depending on what is available on your system
curl -fsSL https://pixi.sh/install.sh | sh
# or
wget -qO- https://pixi.sh/install.sh | sh

# Reload your shell to apply PATH changes
source ~/.bashrc     # use ~/.zshrc if you are on zsh

# Verify installation
pixi --version
# Expected: pixi 0.63.2  (or newer)
```

**VSCode** — Download and install from <https://code.visualstudio.com>. After installation, open the **Extensions** panel (`Ctrl+Shift+X` / `Cmd+Shift+X`) and install the **Python** extension by Microsoft. This provides Jupyter notebook support and Python IntelliSense. Windows users should also install the **WSL** extension to open projects inside WSL2 directly from VSCode.

**NVIDIA GPU (Optional)** — If your machine has an NVIDIA GPU, make sure the driver is installed so that PyTorch can use CUDA. Run `nvidia-smi` to check. If the command is not found, install the driver from <https://www.nvidia.com/en-us/drivers/>. WSL2 users do not need to install the driver inside Linux — the Windows host driver is shared automatically. If you do not have an NVIDIA GPU, PyTorch will use CPU, which is sufficient for all course assignments.

### Step 1: Clone the Course Repository
```bash
git clone --branch students https://github.com/GLI-Lab/machine-learning-course.git
cd machine-learning-course
```

After cloning, the repository structure will look like this:
```bash
machine-learning-course/       # course repository root
├── pixi.toml                  # dependency declarations
├── pixi.lock                  # exact pinned versions
├── data/                      # shared datasets used across labs
├── assignments/               # homeworks for students
│   ├── hw01/
│   ├── hw02/
├── exercises/                 # lab notebooks for students
│   ├── lab01/
│   ├── lab02/
│   └── ...
```

### Step 2: Install All Required Packages

Navigate into the repository folder and run:
```bash
pixi install
```

This reads `pixi.lock` and installs every package at its exact pinned version. The full dependency list is declared in `pixi.toml`:
```{.toml filename="pixi.toml"}
[dependencies]
python = "3.12.*"
numpy = "*"              # numerical computing
pandas = "*"             # data manipulation
scikit-learn = "*"       # machine learning
matplotlib = "*"         # visualization
seaborn = "*"            # visualization
plotly = "*"             # visualization (interactive)
tqdm = "*"               # utils
jupyterlab = "*"         # notebook

[pypi-dependencies]  # linux-64: CUDA 활용
torch = { version = "==2.10.*", index = "https://download.pytorch.org/whl/cu128" }
torchvision = { version = "*", index = "https://download.pytorch.org/whl/cu128" }

[target.osx-arm64.pypi-dependencies]  # osx-arm64: CUDA 없음
torch = { version = "==2.10.*", index = "https://download.pytorch.org/whl/cpu" }
torchvision = { version = "*", index = "https://download.pytorch.org/whl/cpu" }

[tasks]
jupyter = "jupyter lab --ip=0.0.0.0 --port=8888 --no-browser"
gpu-test = 'python -c "import torch; print(torch.__version__); print(torch.cuda.is_available())"'
gpu-info = 'python -c "import torch; [print(f\"GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory // 1024**3}GB\") for i in range(torch.cuda.device_count())]"'

[target.osx-arm64.tasks]
gpu-test = 'python -c "import torch; print(torch.__version__); print(torch.backends.mps.is_available())"'
gpu-info = 'python -c "import subprocess, torch; mem=int(subprocess.check_output([\"sysctl\",\"-n\",\"hw.memsize\"])); print(f\"MPS available: {torch.backends.mps.is_available()}\"); print(f\"Unified Memory: {mem // 1024**3}GB (CPU+GPU shared)\")"'
```

You do not need to install any of these separately — `pixi install` takes care of everything. On macOS (Apple Silicon), pixi automatically selects the CPU-only PyTorch build.

After installation, verify that PyTorch is working:
```bash
pixi run gpu-test
pixi run gpu-info
```

This prints the PyTorch version and whether GPU acceleration is available (`True` for CUDA or MPS, `False` for CPU-only). Either result is fine — GPU is not required for this course.

> #### 📝 How `pixi install` Behaves
>
> When `pixi.lock` exists (the normal case after cloning), pixi installs exact pinned versions with no solver step — this is fast.
>
> When no lock file exists, pixi reads constraints from `pixi.toml`, resolves versions, writes a new `pixi.lock`, then installs. This is slower due to the resolver, but only happens once.
>
> When `pixi.toml` is modified (e.g., after `pixi add <package>`), the lock file is updated automatically on the next `pixi install` or `pixi run`.

### Step 3: Open a Lab Notebook

You can open lab notebooks using either JupyterLab or VSCode. Both work with the pixi environment.

**JupyterLab** — Launch from the repository root:
```bash
pixi run jupyter
```

This starts a JupyterLab server on port 8888. It does not open a browser automatically, so copy the URL printed in your terminal and paste it into your browser. The URL will look like this:
```bash
http://127.0.0.1:8888/lab?token=your_token_here
```

Use this command at the start of every lab session.

**VSCode** — Open the repository folder in VSCode, then select the pixi kernel for your notebooks:

1. Open any `.ipynb` file in VSCode.
2. Click **Select Kernel** in the top-right corner of the notebook editor.
3. Choose **Python Environments**, then select the interpreter located at `.pixi/envs/default/bin/python` inside the repository folder.

> #### 📝 Everyday Workflow
>
> For all regular lab work, `pixi run jupyter` or opening notebooks in VSCode is all you need. If you prefer staying in an activated shell for an extended session, run `pixi shell` to activate the environment directly, then launch tools from there. Exit with `exit` when done.

## Option B (Cloud): Google Colab

Google Colab lets you run any lab notebook directly in your browser with no local installation. To open a lab, click the Colab badge at the top of each notebook:

![](https://colab.research.google.com/assets/colab-badge.svg)

Colab already has most course packages pre-installed, including `numpy`, `matplotlib`, `sklearn`, and recent versions of `torch`. You may need to install any course-specific utilities by running the setup cell at the very top of each notebook, which will look like this:
```{.python}
# Run this cell only if you are on Google Colab
!pip install <course-specific-package>
```

Lab notebooks that use datasets stored in the course repository (under `data/`) require an extra step on Colab. Since Colab cannot access your local file system, clone the course repository at the start of each session:
```{.python}
# Run this cell only if you are on Google Colab
!git clone --branch students https://github.com/GLI-Lab/machine-learning-course.git
```

Each notebook will include this cell already. You just need to run it before proceeding.

> #### ⚠️ Colab Limitations
>
> Colab environments are **ephemeral**: any package you install or file you download is lost when the session ends or times out. You will need to re-run all setup cells at the start of every new session.
>
> For consistent, persistent environments across all labs, the local pixi setup (Option A) is strongly recommended.